# Level-error check (2026-08-23)

One question: **does `G` have a systematic level error in the overlap region?**

Background: the un-stratified `si_sdr` added in `9f62f4b` came back at -13.17 dB
while its own constituents read +46.91 (depth 1) and +1.08 (depth 2) -- below
every depth it appeared to summarise in 271 of 288 rows. Root cause is not a
wiring bug: SI-SDR fits ONE scalar over whatever samples it is handed, so the
near-exact solo copy pins that scalar near 1 and a pure **level** error in the
overlap region is charged at full price, while every per-depth row fits its own
scalar and discounts the same error entirely.

`level_error_db` measures that directly. For an estimate `c*s + n` with `n`
orthogonal to `s` the fitted alpha IS `c`, and depth 1 is a verbatim copy so its
alpha is ~1. So the column reads as **`G`'s systematic output gain error in dB**,
signed: **positive = too loud, negative = too quiet**.

**Cost:** 3 scenes x 2 arms x 1 dilation = 6 units, ~12-15 min of scoring after
~10 min of setup.

**Assertion discipline:** exactly ONE fatal assertion (Cell 6, the reproduction
gate -- if committed numbers moved, everything downstream is invalid). Every
other check diagnoses inline and reports a verdict. In Kaggle batch an uncaught
exception kills every cell after it, and that is what cost us the Test E
diagnosis on 2026-08-23.

In [ ]:
# Cell 1 -- repo + deps + assert the level-error code is actually in the clone.
#
# NOTE: this clones from GitHub, so the 2026-08-23 fix must be PUSHED first.
# The assertions below fail fast if it is not, rather than 25 minutes in.
!git clone -q https://github.com/RohanBanerjee88/dagger.git
import os
os.chdir("dagger")
!pip install -q -e '.[data,ml,dev]'
!pip install -q -U numba numba-cuda
!pip install -q -e '.[diarize]'

import ast, re, subprocess
from pathlib import Path
import yaml

sysmod = Path("dagger/eval/systems.py").read_text()
sisdr = Path("dagger/metrics/sisdr.py").read_text()

assert "si_sdr_pooled_by_depth" in sisdr, "stale clone: no pooled exchange rate"
assert "depth_scale_factors" in sisdr, "stale clone: no per-depth scale factors"
assert "_level_error_db" in sysmod, "stale clone: no level-error column"

def _fields(name):
    m = re.search(rf"^{name} = (\[.*?\])", sysmod, re.S | re.M)
    assert m, f"could not find {name} in dagger/eval/systems.py"
    return ast.literal_eval(m.group(1))

OVER = _fields("OVERALL_FIELDS")
for col in ("si_sdr", "si_sdr_pooled", "level_error_db"):
    assert col in OVER, f"OVERALL_FIELDS is missing {col}: {OVER}"
# The grain guard: a `depth` key would let any depth-stratified table absorb
# these rows as an extra depth.
assert "depth" not in OVER, "OVERALL_FIELDS grew a depth column"
assert "depth" in _fields("SCORE_FIELDS"), "SCORE_FIELDS lost its depth column"

BASE = "configs/phase3/experiments/phase3_librimix_3spk_dilation_v2.yaml"
REF = ("results/phase3/experiments/experiment_stage_B_run_1/"
       "phase3_librimix_3spk_dilation_sweep.csv")
for f in (BASE, REF):
    assert Path(f).exists(), f"missing: {f}"
CKPT = yaml.safe_load(open(BASE))["extractor"]["checkpoint"]

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout
assert "T4" in gpu, f"need T4, got {gpu.strip()}"
print("clone carries the level-error fix")
print(f"  OVERALL_FIELDS: {OVER}")

In [ ]:
# Cell 2 -- HF token, LibriSpeech, LibriMix metadata, env. (Unchanged from the
# 2026-08-23 verification notebook -- proven setup, do not improvise here.)
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

os.environ["DAGGER_HF_TOKEN"] = UserSecretsClient().get_secret("DAGGER_HF_TOKEN")
os.environ["HF_TOKEN"] = os.environ["DAGGER_HF_TOKEN"]
assert os.environ["DAGGER_HF_TOKEN"].startswith("hf_")

!mkdir -p /kaggle/working/data/metadata/Libri3Mix
!ln -sfn /kaggle/input/datasets/victorling/librispeech-clean/LibriSpeech /kaggle/working/data/LibriSpeech
!rm -rf /tmp/LibriMix && git clone -q https://github.com/JorisCos/LibriMix.git /tmp/LibriMix
!cp /tmp/LibriMix/metadata/Libri3Mix/libri3mix_test-clean.csv \
    /kaggle/working/data/metadata/Libri3Mix/libri3mix_test.csv

os.environ["DAGGER_DATA_ROOT"] = "/kaggle/working/data"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("env set")

In [ ]:
# Cell 3 -- long-scene metadata and the clip50 checkpoint.
#
# The corpus MUST be byte-identical to the one behind the committed run-1 CSVs,
# or Cell 6 compares two different corpora and fails for the wrong reason. Same
# flags as every prior long-scene run; the generator is deterministic given
# --seed (confirmed 2026-08-19, and again by the 144/144 smoke on 2026-08-23).
!python scripts/build_long_scene_metadata.py \
    --librispeech-root $DAGGER_DATA_ROOT/LibriSpeech/test-clean \
    --output $DAGGER_DATA_ROOT/metadata/Libri3Mix/libri3mix_test_long.csv \
    --n-src 3 --num-scenes 50 --per-speaker-sec 50 --overlap 0.3

import shutil, torch
from huggingface_hub import hf_hub_download

cached = hf_hub_download(repo_id="AdityaAA2004/dagger-phase2-final-model",
                         filename="phase2_final_model_weights.pt",
                         token=os.environ["DAGGER_HF_TOKEN"])
Path(CKPT).parent.mkdir(parents=True, exist_ok=True)
shutil.copy(cached, CKPT)
meta = torch.load(CKPT, map_location="cpu", weights_only=False)
assert meta.get("system") == "proposed" and meta.get("trained_n_src") == [3, 4, 5]
print(f"checkpoint OK: trained_n_src={meta.get('trained_n_src')}")

In [ ]:
# Cell 4 -- offline suite (~30 s). Expect 475 passed / 1 skipped after the
# 2026-08-23 fix (was 453). A different count means the clone is not what you
# think it is.
!python -m pytest tests/ -q 2>&1 | tail -5

In [ ]:
# Cell 5 -- the run. 3 scenes x 2 arms x 1 dilation = 6 units, ~12-15 min.
#
# 0 ms ONLY: dilation is not the question here, and every sweep point costs a
# full pass. The effective config is WRITTEN BESIDE THE RESULTS -- Stage B run 1
# mutated its config in-kernel and never brought it back, so the committed
# results silently stopped regenerating from the committed configs (§7).
import time, yaml
from pathlib import Path

cfg = yaml.safe_load(open(BASE))
cfg["dataset"]["limit"] = 3
cfg["diarizer"]["dilate_overlap_ms"] = [0]
cfg["eval"] = {"results_dir": "results/verify", "tag": "levelcheck"}

Path("results/verify").mkdir(parents=True, exist_ok=True)
EFFECTIVE = Path("results/verify/effective_levelcheck.yaml")
EFFECTIVE.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(EFFECTIVE.read_text())

t0 = time.time()
!python scripts/run_phase3.py --config {EFFECTIVE}
print(f"[run] {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Cell 6 -- THE ONE FATAL CHECK. `dagger/eval/systems.py` changed, so CLAUDE.md's
# A5 guard applies: did that change move a committed number?
#
# Compared BY VALUE within tolerance, not by bytes. Last night's byte-identity
# "failure" was 5401 CRLF terminators plus a trailing newline with all 5400
# values bit-identical -- a byte comparison across environments tests the csv
# dialect and the CUDA stack as much as it tests the code.
import csv, math

TOL_DB = 1e-3

def keyed(path):
    out = {}
    for r in csv.DictReader(open(path)):
        if float(r["dilate_ms"] or 0.0) != 0.0:
            continue
        out[(r["diarization"], r["scene"], r["speaker"], r["system"], r["depth"])] = \
            float(r["si_sdr"])
    return out

ref = keyed(REF)
new = keyed("results/verify/phase3_librimix_3spk_levelcheck.csv")
shared = set(ref) & set(new)

deltas = {}
for k in shared:
    a, b = ref[k], new[k]
    if a == b:            # covers exact equality including matching +-inf
        continue
    deltas[k] = abs(a - b)
worst = max(deltas.values()) if deltas else 0.0

print(f"[A5] shared rows: {len(shared)}   (expect 144)")
print(f"[A5] rows differing at all: {len(deltas)}")
print(f"[A5] max |delta si_sdr|: {worst:.3e} dB   (tolerance {TOL_DB:g})")
for k in sorted(deltas, key=deltas.get, reverse=True)[:5]:
    print(f"      {k}: {ref[k]} -> {new[k]}")

assert shared, "no shared rows -- the corpus or the grid differs from run 1"
assert worst < TOL_DB, f"per-depth numbers MOVED (max {worst:.3e} dB). STOP."
print("[A5] PASS -- the level-error change left every committed number intact")

In [ ]:
# Cell 7 -- WHAT WE CAME FOR. Diagnoses inline; never aborts.
import csv, math, statistics as st
from collections import defaultdict

ovr = list(csv.DictReader(open(
    "results/verify/phase3_librimix_3spk_levelcheck_overall.csv")))
per = defaultdict(dict)
for r in csv.DictReader(open("results/verify/phase3_librimix_3spk_levelcheck.csv")):
    per[(r["diarization"], r["scene"], r["speaker"], r["system"])][int(r["depth"])] = \
        float(r["si_sdr"])

print(f"overall rows: {len(ovr)}   (expect 72 = 3 scenes x 3 spk x 4 systems x 2 arms)")

print()
print("=== 1. LEVEL ERROR  (signed: + = G too LOUD, - = too quiet) ===")
print(f"{'arm':<7} {'system':<19} {'n':>3} {'median':>8} {'min':>8} {'max':>8}")
lv = defaultdict(list)
for r in ovr:
    v = float(r["level_error_db"])
    if math.isfinite(v):
        lv[(r["diarization"], r["system"])].append(v)
for (arm, sysn), vals in sorted(lv.items()):
    vals = sorted(vals)
    print(f"{arm:<7} {sysn:<19} {len(vals):>3} {st.median(vals):>8.2f} "
          f"{vals[0]:>8.2f} {vals[-1]:>8.2f}")

pool = [v for vals in lv.values() for v in vals]
all_med = st.median(pool) if pool else float("nan")
print(f"\nMEDIAN ACROSS EVERYTHING: {all_med:+.2f} dB"
      f"   ({10 ** (abs(all_med) / 20):.2f}x amplitude)")

print()
print("=== 2. POOLED vs WHOLE TRACK ===")
print(f"{'arm':<7} {'system':<19} {'whole':>8} {'pooled':>8}")
agg = defaultdict(lambda: ([], []))
for r in ovr:
    w, p = float(r["si_sdr"]), float(r["si_sdr_pooled"])
    if math.isfinite(w) and math.isfinite(p):
        agg[(r["diarization"], r["system"])][0].append(w)
        agg[(r["diarization"], r["system"])][1].append(p)
for (arm, sysn), (ws, ps) in sorted(agg.items()):
    print(f"{arm:<7} {sysn:<19} {st.mean(ws):>8.2f} {st.mean(ps):>8.2f}")

print()
print("=== 3. THE BOUND TEST B NEVER ACTUALLY CHECKED ===")
print("(pooled must lie between the best and worst per-depth value; the")
print(" whole-track number has NO such bound and is not tested here)")
checked = viol = 0
for r in ovr:
    d = per[(r["diarization"], r["scene"], r["speaker"], r["system"])]
    fin = [x for x in d.values() if math.isfinite(x)]
    p = float(r["si_sdr_pooled"])
    if len(fin) < 2 or not math.isfinite(p):
        continue
    checked += 1
    if not (min(fin) - 1e-6 <= p <= max(fin) + 1e-6):
        viol += 1
        print(f"  OUT OF BOUNDS: {r['diarization']}/{r['scene'][:11]}/{r['speaker']}"
              f"/{r['system']}: pooled {p:.2f} vs depths {sorted(fin)}")
# Print the count BEFORE the verdict. Test B printed PASS having verified 0 rows.
print(f"speakers checked: {checked}   out of bounds: {viol}")
print("VERDICT:", "PASS" if (checked > 0 and viol == 0)
      else ("VACUOUS -- verified nothing" if checked == 0 else "FAIL"))

In [ ]:
# Cell 8 -- the branch. Which run gets booked next.
THRESHOLD_DB = 6.0   # >2x amplitude: a systematic level error, not estimation noise

print("=" * 72)
if not math.isfinite(all_med):
    print("INCONCLUSIVE -- no finite level_error_db rows. Check Cell 7's input.")
elif abs(all_med) > THRESHOLD_DB:
    direction = "TOO LOUD" if all_med > 0 else "TOO QUIET"
    print(f"LEVEL ERROR IS LARGE: {all_med:+.2f} dB -- G is {direction} by "
          f"{10 ** (abs(all_med) / 20):.2f}x in the overlap region.")
    print()
    print("  This is upstream of every Phase 3 conclusion and is invisible to")
    print("  every scale-invariant metric in the project. It would also corrupt")
    print("  anything NOT scale-invariant: Whisper WER in Phase 4, the noise-term")
    print("  reconstruction loss, any sum-to-mixture check.")
    print()
    print("  -> DO NOT book dilation_v2 (6.7 h). Tuning a mask knob on top of a")
    print("     level bug is premature.")
    print("  -> Book vi_on (5.0 h) in that slot instead, and chase the gain error:")
    print("     candidates are G itself, the Stage-1 normalize/denormalize round")
    print("     trip in _TFGridNetCrossAttnModule.forward, and the w_Oi crossfade.")
else:
    print(f"LEVEL ERROR IS SMALL: {all_med:+.2f} dB. G's output level is fine.")
    print()
    print("  -> Proceed as planned: B2 refine_ceiling (1.7 h), then dilation_v2")
    print("     (6.7 h). Read the sweep on si_sdr_pooled, NOT on si_sdr.")
print("=" * 72)